# Regenerasi Gbr. 2–4 dan Tabel I–III (revisi JOIV)

Gbr. 2–4 di naskah berasal dari run lama (MAPE 1,66) dan tidak cocok dengan Tabel I–III.
Notebook ini menghitung ulang **semua angka di balik Gbr. 2–4 dan Tabel I–III dari satu run yang sama**,
lalu menggambar ulang ketiganya dengan gaya yang sama dengan Gbr. 5–6.

| Keluaran | Isi |
|---|---|
| `TableI_regen.csv` | MAPE satu langkah 8 model terlatih (30 seed) + 3 baseline, SD, 95% CI, MASE |
| `TableII_regen.csv` | MAPE multi-langkah LSTM-PSO h = 1–20 vs RW & RW+drift, DM mentah & Holm |
| `TableIII_regen.csv` | MAPE lintas saham (6 saham, 30 seed), selisih & rasio terhadap RW |
| `Ablation_regen.csv` | Ablasi set fitur (Close, OHLCV, OHLCV+TA, OHLCV+TA+EXT) |
| `Fig2_multistep.png`, `Fig3_diagnostics.png`, `Fig4_crosscutting.png` | Gambar baru, 600 dpi |
| `angka_naskah.json` | Semua angka yang dikutip di teks (DM satu langkah, MASE, uji tanda, dst.) |

**Cara pakai:** GPU T4, Internet ON, Save & Run All (Commit). Kalau log akhir masih menulis
`Dilewati: [...]`, tambahkan output versi ini (dan output notebook E6, agar 15 seed E6c terpakai) sebagai
input, lalu commit lagi. Perkiraan ±6–9 jam GPU total (≈ 450 pelatihan); dengan cache E6 lebih cepat.

In [ ]:
# Yahoo Finance compatibility shim: if yfinance fails (version mismatch such as
# "Impersonating chromeXXX is not supported") or returns nothing, fall back to the Yahoo v8 chart endpoint.
# Prices are dividend/split adjusted, equivalent to auto_adjust=True.
import pandas as pd, requests
try:
    import yfinance as yf
    _yf_download = yf.download
except Exception:
    yf, _yf_download = None, None

def _chart_api(ticker, start, end):
    p1, p2 = int(pd.Timestamp(start).timestamp()), int(pd.Timestamp(end).timestamp())
    for host in ('query1', 'query2'):
        r = requests.get(f'https://{host}.finance.yahoo.com/v8/finance/chart/{ticker}',
                         params=dict(period1=p1, period2=p2, interval='1d', events='div,splits'),
                         headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124 Safari/537.36'}, timeout=30)
        if r.status_code != 200: continue
        res = r.json()['chart']['result'][0]; q = res['indicators']['quote'][0]
        adj = res['indicators'].get('adjclose', [{}])[0].get('adjclose', q['close'])
        d = pd.DataFrame({'Date': pd.to_datetime(res['timestamp'], unit='s'), 'Open': q['open'], 'High': q['high'],
                          'Low': q['low'], 'Close': q['close'], 'Volume': q['volume'], 'Adj': adj}).dropna()
        f = d['Adj'] / d['Close']
        for c in ('Open', 'High', 'Low', 'Close'): d[c] = d[c] * f
        d['Date'] = (d['Date'] + pd.Timedelta(hours=7)).dt.normalize()
        return d.drop(columns='Adj').set_index('Date')
    raise RuntimeError(f'chart API failed for {ticker}')

def _download(ticker, start=None, end=None, **kw):
    try:
        d = _yf_download(ticker, start=start, end=end, **kw) if _yf_download else None
        if d is not None and len(d) > 200: return d
    except Exception:
        pass
    return _chart_api(ticker, start, end)

if yf is not None: yf.download = _download


In [ ]:
# ============================================================================
# SEL 1. KONFIGURASI, ANGGARAN WAKTU, CACHE
# ============================================================================
import os, glob, math, time, pickle, random, warnings, shutil, json
import numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')

T_MULAI = time.time()
TIME_BUDGET_H = 11.0
N_SEED = 30            # naskah: 30 run per model (Tabel I, II, III)
N_SEED_TS = 5          # seed per titik pada sensitivitas timestep (Gbr. 3e)
H = 20                 # horizon multi-langkah
SEED = 42
FINAL_EPOCHS = 100
RUN = dict(T1=True, T2=True, T3=True, ABL=True, TS=True)
SMOKE_TEST = False     # True = versi mini untuk cek sintaks

OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else './out_fig'
CACHE = os.path.join(OUT, 'cache_fig')
os.makedirs(CACHE, exist_ok=True)
_n = 0
for f in (glob.glob('/kaggle/input/**/cache_fig/*.pkl', recursive=True)
          + glob.glob('/kaggle/input/**/cache_e6/e6c_*.pkl', recursive=True)):
    t = os.path.join(CACHE, os.path.basename(f))
    if not os.path.exists(t):
        try: shutil.copy(f, t); _n += 1
        except Exception: pass
print(f'cache dipulihkan: {_n} | total: {len(glob.glob(CACHE + "/*.pkl"))}')

def jam(): return (time.time() - T_MULAI) / 3600
def sisa(): return TIME_BUDGET_H - jam()
DILEWATI = []
def ada(k): return os.path.exists(os.path.join(CACHE, k + '.pkl'))
def disk(k, fn):
    p = os.path.join(CACHE, k + '.pkl')
    if os.path.exists(p):
        with open(p, 'rb') as f: return pickle.load(f)
    v = fn(); tmp = p + f'.tmp{os.getpid()}'
    with open(tmp, 'wb') as f: pickle.dump(v, f)
    os.replace(tmp, p); return v
def boleh(k, perlu=0.3):
    """True bila item k sudah di cache atau waktu masih cukup untuk menghitungnya."""
    if ada(k) or sisa() > perlu: return True
    if k not in DILEWATI: DILEWATI.append(k)
    return False

if SMOKE_TEST:
    N_SEED = 3; N_SEED_TS = 2; FINAL_EPOCHS = 2

# gaya gambar seragam dengan Gbr. 5–6
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8,
                     'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.dpi': 600})
INK, GREY, LIGHT = '#000000', '#7f7f7f', '#c8c8c8'

In [ ]:
# ============================================================================
# SEL 2. DATA, FITUR, MODEL (protokol naskah)
# ============================================================================
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers as KL
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
print('GPU:', tf.config.list_physical_devices('GPU') or 'TIDAK ADA — nyalakan GPU T4')

def set_seed(s): random.seed(s); np.random.seed(s); tf.random.set_seed(s)
TARGET, TRAIN_FRAC = 'Close', 0.8
START, END = '2019-07-01', '2025-07-01'

def _yf(tk, cols=('Open', 'High', 'Low', 'Close', 'Volume')):
    import yfinance as yf
    d = yf.download(tk, start=START, end=END, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex): d.columns = d.columns.get_level_values(0)
    return d.reset_index()[['Date', *cols]].dropna().reset_index(drop=True)

def _fake(seed):
    r = np.random.RandomState(seed); n = 1449; c = 600*np.exp(np.cumsum(r.normal(0, .015, n)))
    return pd.DataFrame({'Date': pd.date_range(START, periods=n, freq='B'), 'Open': c, 'High': c*1.01, 'Low': c*.99,
                         'Close': c, 'Volume': r.randint(1e5, 1e6, n).astype(float)})

def _ema(s, n): return s.ewm(span=n, adjust=False).mean()
def add_technical(df):
    d = df.copy(); c = d.Close
    d['SMA10'] = c.rolling(10).mean(); d['SMA20'] = c.rolling(20).mean(); d['EMA10'] = _ema(c, 10); d['EMA20'] = _ema(c, 20)
    dl = c.diff(); up = dl.clip(lower=0); dn = -dl.clip(upper=0)
    rs = up.ewm(alpha=1/14, adjust=False).mean() / dn.ewm(alpha=1/14, adjust=False).mean(); d['RSI14'] = 100 - 100/(1+rs)
    m = _ema(c, 12) - _ema(c, 26); d['MACD'] = m; d['MACD_sig'] = _ema(m, 9)
    d['ret1'] = c.pct_change(); d['vol10'] = d['ret1'].rolling(10).std(); return d

_EXT = {}
def external():
    if not _EXT:
        for name, tk in {'JKSE': '^JKSE', 'USDIDR': 'USDIDR=X'}.items():
            try:
                if SMOKE_TEST: raise RuntimeError
                _EXT[name] = _yf(tk, cols=('Close',)).rename(columns={'Close': name})
            except Exception:
                _EXT[name] = None; print(f'PERINGATAN: {name} tidak tersedia, diisi nol')
    return _EXT

def load_stock(tk, seed=0):
    raw = _fake(seed) if SMOKE_TEST else _yf(tk)
    d = add_technical(raw)
    for name, e in external().items():
        if e is None: d[name] = 0.0
        else: d = d.merge(e, on='Date', how='left'); d[name] = d[name].ffill().bfill()
    return d.dropna().reset_index(drop=True)

OHLCV = ['Open', 'High', 'Low', 'Close', 'Volume']
TA = ['SMA10', 'SMA20', 'EMA10', 'EMA20', 'RSI14', 'MACD', 'MACD_sig', 'ret1', 'vol10']
EXT = ['JKSE', 'USDIDR']
FEATS = OHLCV + TA + EXT
STOCKS = ['SIDO.JK', 'BBCA.JK', 'BMRI.JK', 'TLKM.JK', 'ASII.JK', 'UNVR.JK']
DATA = {tk: load_stock(tk, i) for i, tk in enumerate(STOCKS)}
df_sido = DATA['SIDO.JK']
print({k.replace('.JK', ''): len(v) for k, v in DATA.items()})

OPTS = ['adam', 'rmsprop', 'nadam']
def _opt(name, lr): return {'adam': Adam, 'rmsprop': RMSprop, 'nadam': Nadam}[name](lr)

class LastStep(KL.Layer):
    def call(self, x): return x[:, -1, :]
class PosEnc(KL.Layer):
    def build(self, shape):
        T, D = int(shape[1]), int(shape[2]); pos = np.arange(T)[:, None]; i = np.arange(D)[None, :]
        ang = pos / np.power(10000, (2*(i//2))/D); pe = np.where(i % 2 == 0, np.sin(ang), np.cos(ang))
        self.pe = tf.constant(pe[None], dtype='float32')
    def call(self, x): return x + self.pe

def build_model(shape, hp, arch='lstm', out=1):
    """LSTM/GRU/BiLSTM: rekuren; TCN: konvolusi kausal berdilatasi; MLP: jendela diratakan; Transformer: 1 blok encoder."""
    K.clear_session(); inp = KL.Input(shape=shape); u = int(hp['units']); dr = float(hp.get('dropout', 0.2))
    if arch in ('lstm', 'gru', 'bilstm'):
        x = inp; L = int(hp.get('n_layers', 1))
        for i in range(L):
            cell = (KL.GRU if arch == 'gru' else KL.LSTM)(u, return_sequences=(i < L-1))
            x = KL.Bidirectional(cell)(x) if arch == 'bilstm' else cell(x)
            if dr > 0: x = KL.Dropout(dr)(x)
    elif arch == 'tcn':
        x = inp
        for dil in (1, 2, 4, 8):
            x = KL.Conv1D(u, 3, padding='causal', dilation_rate=dil, activation='relu')(x)
            if dr > 0: x = KL.Dropout(dr)(x)
        x = LastStep()(x)
    elif arch == 'mlp':
        x = KL.Flatten()(inp); x = KL.Dense(u, activation='relu')(x)
        if dr > 0: x = KL.Dropout(dr)(x)
        x = KL.Dense(u, activation='relu')(x)
    elif arch == 'transformer':
        x = PosEnc()(KL.Dense(u)(inp))
        a = KL.MultiHeadAttention(num_heads=4, key_dim=max(1, u//4), dropout=dr)(x, x)
        x = KL.LayerNormalization()(x + a)
        f = KL.Dense(u, activation='relu')(x); f = KL.Dense(u)(f)
        x = KL.LayerNormalization()(x + f); x = KL.GlobalAveragePooling1D()(x)
        if dr > 0: x = KL.Dropout(dr)(x)
    m = Model(inp, KL.Dense(out, activation='linear')(x))
    m.compile(loss='mse', optimizer=_opt(hp.get('optimizer', 'adam'), hp.get('lr', 1e-3))); return m

def sequences(df, feats, ts, horizon=1):
    """Masukan berakhir di i-1, target y[i..i+horizon-1]. Skala dicocokkan pada porsi latih saja."""
    dX = df[feats].values.astype('float32'); dy = df[[TARGET]].values.astype('float32')
    n = len(df); n_tr = int(n*TRAIN_FRAC)
    sx = MinMaxScaler().fit(dX[:n_tr]); sy = MinMaxScaler().fit(dy[:n_tr])
    Xs, ys = sx.transform(dX), sy.transform(dy).ravel()
    idx = np.arange(ts, n - horizon + 1)
    X = np.stack([Xs[i-ts:i] for i in idx]); Y = np.stack([ys[i:i+horizon] for i in idx])
    tr = idx + horizon - 1 < n_tr          # target latih tidak menyentuh porsi uji
    te = idx >= n_tr
    return X[tr], Y[tr], X[te], idx[te], sy

def fit_predict(hp, seed, df=None, feats=None, arch='lstm', horizon=1, keep_hist=False):
    df = df_sido if df is None else df; feats = FEATS if feats is None else feats
    set_seed(seed)
    Xtr, Ytr, Xte, ite, sy = sequences(df, feats, int(hp['timestep']), horizon)
    m = build_model(Xtr.shape[1:], hp, arch, out=horizon)
    h = m.fit(Xtr, Ytr, validation_split=0.2, shuffle=False, epochs=FINAL_EPOCHS, batch_size=int(hp['batch_size']),
              callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
    p = m.predict(Xte, verbose=0).reshape(len(Xte), horizon)
    pred = sy.inverse_transform(p.reshape(-1, 1)).reshape(p.shape)
    y = df[TARGET].values.astype('float64')
    if horizon == 1:
        out = dict(pred=pred[:, 0], y=y[ite], rw=y[ite-1])
    else:
        out = dict(pred=pred, y=np.stack([y[ite+k] for k in range(horizon)], 1), last=y[ite-1])
    if keep_hist: out['hist'] = {k: list(map(float, v)) for k, v in h.history.items()}
    return out

def mape(y, p): return float(np.mean(np.abs((y-p)/y))*100)
def mase(y, p, ytr): return float(np.mean(np.abs(y-p)) / np.mean(np.abs(np.diff(ytr))))
def dm_test(e1, e2, h=1):
    d = np.abs(e1) - np.abs(e2); n = len(d); db = d.mean()
    g = [np.sum((d[k:]-db)*(d[:n-k]-db))/n for k in range(h)]
    v = (g[0] + 2*sum(g[1:])) / n
    dm = db/math.sqrt(v) * math.sqrt((n + 1 - 2*h + h*(h-1)/n)/n)
    return float(dm), float(2*stats.t.sf(abs(dm), df=n-1))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); a = np.empty(m); r = 0.0
    for k, i in enumerate(o): r = max(r, min(1.0, (m-k)*p[i])); a[i] = r
    return a
def ci95(a):
    a = np.asarray(a, float); h = stats.t.ppf(.975, len(a)-1)*a.std(ddof=1)/math.sqrt(len(a)); return a.mean()-h, a.mean()+h

# konfigurasi (Tabel I naskah). Referensi: konfigurasi default bersama.
DEF = dict(units=64, n_layers=1, dropout=0.2, lr=1e-3, batch_size=32, timestep=20, optimizer='adam')
GA_HP = dict(units=126, n_layers=1, dropout=0.140, lr=5.261e-4, batch_size=61, timestep=22, optimizer='rmsprop')
PSO_HP = dict(units=160, n_layers=1, dropout=0.1, lr=1e-3, batch_size=16, timestep=21, optimizer='nadam')
MODELS = [('GRU', DEF, 'gru'), ('LSTM (standard)', DEF, 'lstm'), ('BiLSTM', DEF, 'bilstm'),
          ('LSTM-GA', GA_HP, 'lstm'), ('LSTM-PSO', PSO_HP, 'lstm'), ('TCN', DEF, 'tcn'),
          ('MLP', DEF, 'mlp'), ('Transformer', DEF, 'transformer')]

def cfgkey(hp):   # sama dengan kunci E6c -> 15 seed E6c terpakai ulang
    return '_'.join(map(str, (int(hp['units']), int(hp.get('n_layers', 1)), round(hp['dropout'], 2),
                              round(math.log10(hp['lr']), 2), int(hp['batch_size']), int(hp['timestep']), hp['optimizer'])))

def one_step_runs(hp, arch, df=None, feats=None, tag='sido', n=None):
    n = N_SEED if n is None else n; outs = []
    for s in range(n):
        k = (f'e6c_{arch}_{cfgkey(hp)}_s{s}' if tag == 'sido' and feats is None
             else f'os_{tag}_{arch}_{cfgkey(hp)}_s{s}')
        if not boleh(k): break
        outs.append(disk(k, lambda s=s: fit_predict(hp, SEED + s, df=df, feats=feats, arch=arch)))
    return outs

In [ ]:
# ============================================================================
# SEL 3. TABEL I — satu langkah, 8 model terlatih x N_SEED + 3 baseline
# ============================================================================
n_tr = int(len(df_sido)*TRAIN_FRAC); y_all = df_sido.Close.values.astype(float)
y_te, rw_te = y_all[n_tr:], y_all[n_tr-1:-1]; drift = np.mean(np.diff(y_all[:n_tr]))
NAIVE = mape(y_te, rw_te)

def arima_rolling():
    if SMOKE_TEST: return rw_te.copy()
    from statsmodels.tsa.arima.model import ARIMA
    pr = []
    for t in range(n_tr, len(y_all)):
        pr.append(float(ARIMA(y_all[:t], order=(1, 1, 1)).fit().forecast(1)[0]))
    return np.array(pr)
arima_pred = disk('arima111_sido', arima_rolling)

T1 = []; RUNS = {}
for lbl, val in (('Naïve random walk', rw_te), ('Naïve + drift', rw_te + drift), ('ARIMA(1,1,1)', arima_pred)):
    T1.append(dict(Model=lbl, n=1, MAPE=mape(y_te, val), SD=0.0, CI_lo=np.nan, CI_hi=np.nan, MASE=mase(y_te, val, y_all[:n_tr])))
if RUN['T1']:
    for lbl, hp, arch in MODELS:
        outs = one_step_runs(hp, arch)
        if not outs: continue
        RUNS[lbl] = outs
        a = np.array([mape(o['y'], o['pred']) for o in outs]); lo, hi = ci95(a) if len(a) > 1 else (np.nan, np.nan)
        pm = np.mean([o['pred'] for o in outs], 0)
        T1.append(dict(Model=lbl, n=len(a), MAPE=a.mean(), SD=a.std(ddof=1) if len(a) > 1 else np.nan, CI_lo=lo, CI_hi=hi,
                       MASE=np.mean([mase(o['y'], o['pred'], y_all[:n_tr]) for o in outs])))
        print(f'  {lbl:16s} n={len(a):2d} MAPE {a.mean():.3f} ± {a.std(ddof=1) if len(a)>1 else 0:.3f}')
T1 = pd.DataFrame(T1)
T1_sorted = pd.concat([T1.iloc[:3], T1.iloc[3:].sort_values('MAPE')])
T1_sorted.round(4).to_csv(f'{OUT}/TableI_regen.csv', index=False)
print('\n' + T1_sorted.round(3).to_string(index=False))

# DM satu langkah LSTM-PSO vs RW (seed-averaged) — angka yang dikutip di Bagian III-B
NUM = {'naive_MAPE': NAIVE}
if 'LSTM-PSO' in RUNS:
    pm = np.mean([o['pred'] for o in RUNS['LSTM-PSO']], 0); d_, p_ = dm_test(y_te - pm, y_te - rw_te)
    NUM.update(LSTM_PSO_DM_onestep=d_, LSTM_PSO_DM_p=p_, LSTM_PSO_MASE_mean=float(T1.set_index('Model').loc['LSTM-PSO', 'MASE']))

In [ ]:
# ============================================================================
# SEL 4. TABEL II & GBR. 2 — multi-langkah langsung (H = 20), LSTM-PSO, N_SEED run
#   DM per horizon pada prakiraan rata-rata seed, lag HLN = h, Holm atas 20 horizon.
# ============================================================================
T2 = pd.DataFrame()
if RUN['T2']:
    ms = []
    for s in range(N_SEED):
        k = f'ms_{cfgkey(PSO_HP)}_H{H}_s{s}'
        if not boleh(k): break
        ms.append(disk(k, lambda s=s: fit_predict(PSO_HP, SEED + s, horizon=H)))
    if ms:
        Y, last = ms[0]['y'], ms[0]['last']; P = np.stack([o['pred'] for o in ms])        # (seed, origin, h)
        rows = []
        for h in range(1, H+1):
            yt = Y[:, h-1]; rwp = last; drp = last + h*drift
            per = np.array([mape(yt, P[s][:, h-1]) for s in range(len(ms))])
            pm = P[:, :, h-1].mean(0); d_, p_ = dm_test(yt - pm, yt - rwp, h=h)
            rows.append(dict(h=h, model_mean=per.mean(), model_sd=per.std(ddof=1) if len(per) > 1 else np.nan,
                             naive=mape(yt, rwp), naive_drift=mape(yt, drp), DM=d_, p_raw=p_))
        T2 = pd.DataFrame(rows); T2['p_holm'] = holm(T2.p_raw)
        T2['verdict'] = np.where(T2.p_holm >= .05, 'n.s.', np.where(T2.DM > 0, 'naïve', 'model'))
        T2.round(4).to_csv(f'{OUT}/TableII_regen.csv', index=False)
        print(T2.round(3).to_string(index=False))

        fig, ax = plt.subplots(figsize=(3.45, 2.5))
        x = T2.h.values
        ax.fill_between(x, T2.model_mean - T2.model_sd, T2.model_mean + T2.model_sd, color=LIGHT, lw=0)
        ax.plot(x, T2.model_mean, '-o', color=INK, ms=3, lw=1.3, label=f'LSTM-PSO (mean ± SD, {len(ms)} runs)')
        ax.plot(x, T2.naive, '-', color=GREY, lw=1.3, label='Random walk')
        ax.plot(x, T2.naive_drift, '--', color=GREY, lw=1.1, label='Random walk + drift')
        ax.set_xlabel('Forecast horizon (trading days)'); ax.set_ylabel('Test MAPE (%)')
        ax.set_xticks([1, 5, 10, 15, 20]); ax.set_ylim(0, None); ax.grid(axis='y', color='0.9', lw=.6)
        ax.legend(frameon=False, loc='lower right')
        fig.tight_layout(); fig.savefig(f'{OUT}/Fig2_multistep.png', bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================================
# SEL 5. TABEL III & ABLASI — lintas saham (konfigurasi LSTM-PSO dipindahkan apa adanya)
# ============================================================================
T3 = pd.DataFrame()
if RUN['T3']:
    rows = []
    for tk in STOCKS:
        d = DATA[tk]; ntr = int(len(d)*TRAIN_FRAC); yy = d.Close.values.astype(float)
        outs = RUNS.get('LSTM-PSO') if tk == 'SIDO.JK' else one_step_runs(PSO_HP, 'lstm', df=d, feats=FEATS, tag=tk.replace('.JK', ''))
        if not outs: continue
        a = np.array([mape(o['y'], o['pred']) for o in outs]); nv = mape(yy[ntr:], yy[ntr-1:-1])
        rows.append(dict(Stock=tk.replace('.JK', ''), n=len(a), MAPE_mean=a.mean(), MAPE_sd=a.std(ddof=1) if len(a) > 1 else np.nan,
                         naive=nv, gap_pp=a.mean() - nv, ratio=a.mean()/nv))
    T3 = pd.DataFrame(rows).sort_values('MAPE_mean')
    T3.round(4).to_csv(f'{OUT}/TableIII_regen.csv', index=False); print(T3.round(3).to_string(index=False))
    if len(T3) == 6:
        k_ = int((T3.gap_pp > 0).sum())
        NUM.update(sign_test_p=float(stats.binomtest(k_, 6, .5).pvalue), RW_better_count=k_,
                   median_ratio=float(T3.ratio.median()), ratio_min=float(T3.ratio.min()), ratio_max=float(T3.ratio.max()))

ABL = pd.DataFrame()
if RUN['ABL']:
    rows = []
    for nm, fs in (('Close only', ['Close']), ('OHLCV', OHLCV), ('OHLCV + TA', OHLCV + TA), ('OHLCV + TA + EXT', FEATS)):
        outs = RUNS.get('LSTM-PSO') if nm == 'OHLCV + TA + EXT' else one_step_runs(PSO_HP, 'lstm', feats=fs, tag='abl' + str(len(fs)))
        if outs:
            a = np.array([mape(o['y'], o['pred']) for o in outs])
            rows.append(dict(feature_set=nm, n_features=len(fs), n=len(a), MAPE_mean=a.mean(), MAPE_sd=a.std(ddof=1) if len(a) > 1 else np.nan))
    ABL = pd.DataFrame(rows); ABL.round(4).to_csv(f'{OUT}/Ablation_regen.csv', index=False); print(ABL.round(3).to_string(index=False))

In [ ]:
# ============================================================================
# SEL 6. GBR. 3 — diagnostik LSTM-PSO (5 panel)
#   (a) MAPE satu langkah semua model vs baseline (95% CI)  (b) sebaran MAPE antar-run
#   (c) residual: vs prediksi, histogram, Q-Q               (d) kurva loss latih/validasi
#   (e) sensitivitas timestep (konfigurasi LSTM-PSO, N_SEED_TS seed per titik)
# ============================================================================
TS_GRID = [5, 10, 15, 20, 25, 30]
TS = []
if RUN['TS']:
    for ts in TS_GRID:
        hp = dict(PSO_HP, timestep=ts); outs = one_step_runs(hp, 'lstm', n=N_SEED_TS, tag='ts') if ts != PSO_HP['timestep'] else None
        if ts == PSO_HP['timestep'] and 'LSTM-PSO' in RUNS: outs = RUNS['LSTM-PSO'][:N_SEED_TS]
        if outs:
            a = np.array([mape(o['y'], o['pred']) for o in outs]); TS.append(dict(timestep=ts, mean=a.mean(), sd=a.std(ddof=1) if len(a) > 1 else 0))
TS = pd.DataFrame(TS)
curve = disk(f'lc_{cfgkey(PSO_HP)}', lambda: fit_predict(PSO_HP, SEED, keep_hist=True)) if boleh(f'lc_{cfgkey(PSO_HP)}') else None

if 'LSTM-PSO' in RUNS:
    fig = plt.figure(figsize=(7.1, 6.6))
    gs = fig.add_gridspec(3, 6, height_ratios=[1.1, 1, 1], hspace=.75, wspace=1.1)
    learned = T1.iloc[3:].sort_values('MAPE'); base = T1.iloc[:3]
    # (a)
    ax = fig.add_subplot(gs[0, :3]); labs = list(base.Model) + list(learned.Model)
    vals = list(base.MAPE) + list(learned.MAPE)
    err = [[0]*3 + list(learned.MAPE - learned.CI_lo), [0]*3 + list(learned.CI_hi - learned.MAPE)]
    cols = [LIGHT]*3 + [INK if m == 'LSTM-PSO' else '0.5' for m in learned.Model]
    ax.bar(range(len(vals)), vals, color=cols, yerr=err, capsize=2, error_kw=dict(lw=.8))
    ax.axhline(NAIVE, color=INK, lw=.9, ls='--'); ax.text(.02, NAIVE, f'Random walk {NAIVE:.2f}', transform=ax.get_yaxis_transform(), va='bottom', ha='left', fontsize=6.3)
    ax.set_xticks(range(len(vals))); ax.set_xticklabels([{'Naïve random walk': 'Random walk', 'Naïve + drift': 'RW + drift'}.get(l, l.replace(' (standard)', '')) for l in labs], rotation=55, ha='right', fontsize=6.5)
    ax.set_ylabel('Test MAPE (%)'); ax.set_title('(a) One-step MAPE (95% CI)', loc='left')
    # (b)
    ax = fig.add_subplot(gs[0, 3:]); order = list(learned.Model)
    ax.boxplot([[mape(o['y'], o['pred']) for o in RUNS[m]] for m in order], widths=.55, showfliers=True,
               medianprops=dict(color=INK), flierprops=dict(marker='o', ms=2, mfc=GREY, mec='none'))
    ax.axhline(NAIVE, color=INK, lw=.9, ls='--'); ax.set_xticks(range(1, len(order)+1))
    ax.set_xticklabels([m.replace(' (standard)', '') for m in order], rotation=55, ha='right', fontsize=6.5)
    ax.set_ylabel('Test MAPE (%)'); ax.set_title(f'(b) Run-to-run MAPE ({N_SEED} runs)', loc='left')
    # (c) residual seed dengan MAPE median
    o_med = sorted(RUNS['LSTM-PSO'], key=lambda o: mape(o['y'], o['pred']))[len(RUNS['LSTM-PSO'])//2]
    res = o_med['y'] - o_med['pred']
    ax = fig.add_subplot(gs[1, :2]); ax.scatter(o_med['pred'], res, s=4, color=INK, alpha=.6, lw=0); ax.axhline(0, color=GREY, lw=.8)
    ax.set_xlabel('Predicted price (IDR)'); ax.set_ylabel('Residual (IDR)'); ax.set_title('(c) Residuals vs predicted', loc='left')
    ax = fig.add_subplot(gs[1, 2:4]); ax.hist(res, bins=30, color='0.55', edgecolor='white', lw=.4)
    ax.axvline(res.mean(), color=INK, lw=.9); ax.set_xlabel('Residual (IDR)'); ax.set_ylabel('Count')
    ax.set_title(f'Residual mean {res.mean():.1f}', loc='left', fontsize=7.5)
    ax = fig.add_subplot(gs[1, 4:]); (osm, osr), (sl, ic, _) = stats.probplot(res)
    ax.scatter(osm, osr, s=4, color=INK, lw=0); ax.plot(osm, sl*osm + ic, color=GREY, lw=.9)
    ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Ordered residuals'); ax.set_title('Normal Q-Q', loc='left', fontsize=7.5)
    # (d)
    ax = fig.add_subplot(gs[2, :3])
    if curve and 'hist' in curve:
        ax.plot(range(1, len(curve['hist']['loss'])+1), curve['hist']['loss'], color=INK, lw=1.2, label='Training loss')
        ax.plot(range(1, len(curve['hist']['val_loss'])+1), curve['hist']['val_loss'], '--', color=GREY, lw=1.2, label='Validation loss')
        ax.legend(frameon=False)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (scaled)'); ax.set_title('(d) Learning curve (seed 42)', loc='left')
    # (e)
    ax = fig.add_subplot(gs[2, 3:])
    if len(TS):
        ax.errorbar(TS.timestep, TS['mean'], yerr=TS['sd'], fmt='-o', color=INK, ms=3, capsize=2, lw=1.1, label=f'LSTM-PSO ({N_SEED_TS} runs)')
    ax.axhline(NAIVE, color=INK, lw=.9, ls='--', label='Random walk')
    ax.set_xlabel('Timestep (window length)'); ax.set_ylabel('Test MAPE (%)'); ax.set_ylim(0, None)
    ax.set_title('(e) Timestep sensitivity', loc='left'); ax.legend(frameon=False)
    for a_ in fig.axes: a_.grid(axis='y', color='0.92', lw=.5); a_.set_axisbelow(True)
    fig.savefig(f'{OUT}/Fig3_diagnostics.png', bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================================
# SEL 7. GBR. 4 — (a) DM p-value vs horizon, (b) peta selisih MAPE lintas saham
# ============================================================================
if len(T2) and len(T3):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.1, 2.7), gridspec_kw=dict(width_ratios=[1.15, 1], wspace=.35))
    sig = T2[(T2.p_holm < .05) & (T2.DM > 0)].h.values
    for h in sig: a1.axvspan(h-.5, h+.5, color='0.9', lw=0)
    a1.semilogy(T2.h, T2.p_raw.clip(lower=1e-6), '-o', color=INK, ms=3, lw=1.2, label='DM p (raw)')
    a1.semilogy(T2.h, T2.p_holm.clip(lower=1e-6), '--s', color=GREY, ms=3, lw=1.1, label='DM p (Holm-adjusted)')
    a1.axhline(.05, color=INK, lw=.7, ls=':'); a1.text(.6, .05, 'α = 0.05', va='bottom', ha='left', fontsize=6.3)
    a1.set_xticks([1, 5, 10, 15, 20]); a1.set_xlabel('Forecast horizon (trading days)'); a1.set_ylabel('Diebold–Mariano p-value (log)')
    a1.set_title('(a) Significance versus horizon', loc='left'); a1.legend(frameon=False, loc='center right')
    if len(sig):
        a1.text(.02, .03, f'shaded: random walk significantly\nbetter (Holm), h = {", ".join(map(str, sig))}',
                transform=a1.transAxes, va='bottom', fontsize=6.3)
    t = T3.sort_values('gap_pp')
    a2.barh(t.Stock, t.gap_pp, color=[INK if g > 0 else GREY for g in t.gap_pp], height=.6)
    a2.axvline(0, color=INK, lw=.8)
    for yv, (g, m_, n_) in enumerate(zip(t.gap_pp, t.MAPE_mean, t.naive)):
        a2.text(g + t.gap_pp.abs().max()*.02, yv, f'+{g:.2f} pp ({m_:.2f}/{n_:.2f})' if g > 0 else f'{g:.2f} pp', va='center', fontsize=6.3)
    a2.set_xlim(min(0, t.gap_pp.min()*1.1), t.gap_pp.max()*1.55)
    a2.set_xlabel('MAPE gap: LSTM-PSO − random walk (pp)'); a2.set_title('(b) Cross-stock baseline-relative error', loc='left')
    a2.spines['left'].set_visible(False); a2.tick_params(axis='y', length=0)
    fig.savefig(f'{OUT}/Fig4_crosscutting.png', bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================================
# SEL 8. RINGKASAN ANGKA UNTUK NASKAH
# ============================================================================
if len(T2):
    NUM['multistep_sig_horizons_RW_better'] = T2[(T2.p_holm < .05) & (T2.DM > 0)].h.tolist()
    NUM['multistep_model_better_any'] = bool(((T2.p_holm < .05) & (T2.DM < 0)).any())
if len(T1) > 3:
    NUM['best_learned'] = T1.iloc[3:].sort_values('MAPE').iloc[0].Model
json.dump({k: (round(v, 4) if isinstance(v, float) else v) for k, v in NUM.items()}, open(f'{OUT}/angka_naskah.json', 'w'), indent=1, ensure_ascii=False)
print(json.dumps(NUM, indent=1, ensure_ascii=False, default=float))
print(f'\nTerpakai {jam():.2f} jam | Dilewati: {DILEWATI[:5]}{" ..." if len(DILEWATI) > 5 else ""} ({len(DILEWATI)} item)'
      if DILEWATI else f'\nTerpakai {jam():.2f} jam | Dilewati: tidak ada — semua tabel & gambar lengkap')